# Turbulence Benchmark Database Builder

The Turbulence benchmark is used to evaluate the robustness of LLMs in code generation tasks. This notebook uses the source code from the benchmark and adapts it for the use case of testing with MuCoCo.

In [1]:
import os
import random
from typing import Iterable
from tqdm import tqdm
import sys

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from utility.helper_functions import TurbulenceBenchmarkHelper
from database import MongoDBHelper

# Connecting to MongoDB

In [4]:
db = MongoDBHelper(max_retries= 5)
if db.check_database_connectivity():
    print("MongoDB connected")

base_qns_db = db.client["Baseline_Questions_DB"]
baseline_db = base_qns_db["Turbulence_Benchmark"]

MongoDB connected


In [5]:
curr_dir = os.getcwd()
source_code_dir = os.path.join(curr_dir, "Source_Code")
qn_folders = [f for f in os.listdir(source_code_dir) if os.path.isdir(os.path.join(source_code_dir, f))]

## Processing Turbulence Dataset and storing into MongoDB

In [6]:
seed = 1234
random.seed(seed)

failed_tasks = []
for qn_folder_name in tqdm(qn_folders):
    # setting the correct qn_num
    q_no = qn_folder_name.split("Q")[-1]  
    if q_no != "":
        
        func_name = None            # stores the task function name
        
        # obtaining the folder directory to the target qn. In this benchmark, each question is kept in an individual folder.
        qn_folder_dir = os.path.join(source_code_dir, qn_folder_name)
        
        # initializing a TurbulenceBenchmarkHelper object with the question number and seed
        helper = TurbulenceBenchmarkHelper(q_no= q_no, seed=seed)

        # 1. Generating the params to substitute into the solutions etc. 
        gen_params_res = helper.run_gen_params(
            qn_folder_dir = qn_folder_dir,
        )

        # Ensuring that each parameter generated is iterable, else it is converted to a Tuple.
        # This step is necessary as some test functions require an iterable input
        gen_params_res = [(param, ) if not isinstance(param, Iterable) else param for param in gen_params_res]

        # 2. Using the generated params to generate the function inputs.
        input_generator_res = helper.run_input_generator(
            qn_folder_dir = qn_folder_dir,
            gen_params = gen_params_res
        )

        # 3. Obtaining the solution, natural language and test templates
        sol_template = TurbulenceBenchmarkHelper.return_template_contents(
            dir = os.path.join(qn_folder_dir, "solution.py.template")
        )

        prompt_template = TurbulenceBenchmarkHelper.return_template_contents(
            dir = os.path.join(qn_folder_dir, "question.txt.template")
        )

        tests_template = TurbulenceBenchmarkHelper.return_template_contents(
            dir = os.path.join(qn_folder_dir, "tests.py.template")
        )

        # Removing the unnecessary import statements in test template
        tests_template = helper.process_test_cases(test_template=tests_template)

        # iterating through each sample generated for each task
        # this step involves iterating through each 
        for idx in range(len(input_generator_res)):
            params = gen_params_res[idx]
            func_input = input_generator_res[idx]

            solution = sol_template
            prompt = prompt_template
            tests = tests_template

            for param_idx, param in enumerate(params):
                solution = solution.replace(f"${param_idx}", str(param))
                tests = tests.replace(f"${param_idx}", str(param))
            
            if func_name is None:
                try:
                    func_name = helper.obtain_func_name(
                        sol_template= solution,
                        qn_txt_template= prompt_template
                        )
                except Exception as e:
                    print(e)
                    failed_tasks.append((q_no, e))
                    continue
            
            
            tests = helper.replace_func_name(tests_template = tests, func_name = func_name)

            try:
                helper.run_test_suite(tests = tests, solution = solution, func_name= func_name)
            except Exception as e:
                print(f'Q{q_no} failed the tests and will not be added to the Turbulence database')
                failed_tasks.append(q_no)
                break
        
        # If statement checking if the question has failed the checks. If so, continue to the next task and do not store this task in the database. 
        if q_no in failed_tasks:
            continue

        # Parameter and test inputs dictionary
        param_dict = {}
        for idx in range(len(gen_params_res)):
            params_entry = gen_params_res[idx]
            input_entry = input_generator_res[idx]

            param_dict[str(idx)] = {
                "params": helper.process_for_mongo_db_storage(params_entry),
                "func_input" : helper.process_for_mongo_db_storage(input_entry)
            }
        
        
        task_id = f"TurbulenceQ{baseline_db.count_documents({}) + 1}"

        # Creating the database entry
        database_entry = {
            "_id": task_id,
            "question_template": tests_template,
            "prompt_template": prompt_template,
            "solution_template": sol_template,
            "func_name": func_name,
            "params": param_dict,
            "original_id": f"Q{q_no}"
        }
        
        # Storing entry into the baseline database
        exisiting_entry = baseline_db.find_one({"_id": task_id})

        try:
            if exisiting_entry is not None:
                baseline_db.find_one_and_replace({"_id": task_id}, database_entry)
            else:
                baseline_db.insert_one(database_entry)
        except Exception as e:
            print(e)
            continue    

 12%|█▏        | 7/60 [00:00<00:00, 68.59it/s]

Q37 failed the tests and will not be added to the Turbulence database
Q30 failed the tests and will not be added to the Turbulence database
Q39 failed the tests and will not be added to the Turbulence database
Q55 failed the tests and will not be added to the Turbulence database
Q52 failed the tests and will not be added to the Turbulence database
Q38 failed the tests and will not be added to the Turbulence database
Q31 failed the tests and will not be added to the Turbulence database
Q36 failed the tests and will not be added to the Turbulence database
Q53 failed the tests and will not be added to the Turbulence database
Q54 failed the tests and will not be added to the Turbulence database
Q49 failed the tests and will not be added to the Turbulence database
Q8 failed the tests and will not be added to the Turbulence database
Q6 failed the tests and will not be added to the Turbulence database
Q47 failed the tests and will not be added to the Turbulence database
Q1 failed the tests an

 35%|███▌      | 21/60 [00:00<00:00, 103.40it/s]

Q25 failed the tests and will not be added to the Turbulence database
Q22 failed the tests and will not be added to the Turbulence database
Q41 failed the tests and will not be added to the Turbulence database
Q7 failed the tests and will not be added to the Turbulence database
Q46 failed the tests and will not be added to the Turbulence database
Q48 failed the tests and will not be added to the Turbulence database
Q9 failed the tests and will not be added to the Turbulence database
Q23 failed the tests and will not be added to the Turbulence database
Q24 failed the tests and will not be added to the Turbulence database
Q12 failed the tests and will not be added to the Turbulence database
Q15 failed the tests and will not be added to the Turbulence database


 58%|█████▊    | 35/60 [00:00<00:00, 106.79it/s]

Q51 failed the tests and will not be added to the Turbulence database
Q56 failed the tests and will not be added to the Turbulence database
Q60 failed the tests and will not be added to the Turbulence database
Q58 failed the tests and will not be added to the Turbulence database
Q33 failed the tests and will not be added to the Turbulence database
Q34 failed the tests and will not be added to the Turbulence database
Q59 failed the tests and will not be added to the Turbulence database
Q57 failed the tests and will not be added to the Turbulence database
Q50 failed the tests and will not be added to the Turbulence database
Q35 failed the tests and will not be added to the Turbulence database
Q32 failed the tests and will not be added to the Turbulence database
Q10 failed the tests and will not be added to the Turbulence database
Q17 failed the tests and will not be added to the Turbulence database
Q28 failed the tests and will not be added to the Turbulence database
Q21 failed the tests

 87%|████████▋ | 52/60 [00:00<00:00, 129.43it/s]

Q26 failed the tests and will not be added to the Turbulence database
Q19 failed the tests and will not be added to the Turbulence database
Q2 failed the tests and will not be added to the Turbulence database
Q43 failed the tests and will not be added to the Turbulence database
Q5 failed the tests and will not be added to the Turbulence database
Q44 failed the tests and will not be added to the Turbulence database
Q27 failed the tests and will not be added to the Turbulence database
Q18 failed the tests and will not be added to the Turbulence database
Q20 failed the tests and will not be added to the Turbulence database
Q16 failed the tests and will not be added to the Turbulence database
Q29 failed the tests and will not be added to the Turbulence database


100%|██████████| 60/60 [00:00<00:00, 117.17it/s]

Q11 failed the tests and will not be added to the Turbulence database
Q4 failed the tests and will not be added to the Turbulence database
Q45 failed the tests and will not be added to the Turbulence database
Q3 failed the tests and will not be added to the Turbulence database
Q42 failed the tests and will not be added to the Turbulence database


### Printing all tasks that failed

In [7]:
print(f"There are a total of {len(failed_tasks)}/60 total tasks were not added to the Turbulence database")
print("The following tasks failed the tests:")
for t in failed_tasks:
    print(f"    Q{t}")

There are a total of 60/60 total tasks were not added to the Turbulence database
The following tasks failed the tests:
    Q37
    Q30
    Q39
    Q55
    Q52
    Q38
    Q31
    Q36
    Q53
    Q54
    Q49
    Q8
    Q6
    Q47
    Q1
    Q40
    Q14
    Q13
    Q25
    Q22
    Q41
    Q7
    Q46
    Q48
    Q9
    Q23
    Q24
    Q12
    Q15
    Q51
    Q56
    Q60
    Q58
    Q33
    Q34
    Q59
    Q57
    Q50
    Q35
    Q32
    Q10
    Q17
    Q28
    Q21
    Q26
    Q19
    Q2
    Q43
    Q5
    Q44
    Q27
    Q18
    Q20
    Q16
    Q29
    Q11
    Q4
    Q45
    Q3
    Q42


## Retrieving documents stored in Turbulence datasets and checking test oracle

This is a necessary step as it ensures that the experiment is still replicable and valid even after storing it into MongoDB. Any tests that fails this step should be removed from the MongoDB database as it is deemed an invalid test set.

In [ ]:
num_tasks = baseline_db.count_documents({})
seed = 1234

for idx in tqdm(range(num_tasks)):

    q_no = idx + 1
    if q_no != "":
        task_id = f"TurbulenceQ{q_no}"

        doc = baseline_db.find_one({"_id": task_id})

        if doc is None:
            print(f"Could not retrieve {task_id} from the dataset.")
            continue

        qn_template = doc["question_template"]
        solution_template = doc["solution_template"]
        func_name = doc["func_name"]
        param_dict: dict = doc["params"]
        original_id: str = doc['original_id']

        original_q_no = original_id.split("Q")[-1]

        for task in param_dict.values():
            params = task['params']
            solution = solution_template
            tests = qn_template

            helper = TurbulenceBenchmarkHelper(q_no= original_q_no, seed = seed)

            processed_params = helper.convert_data_to_metadata(data = params["data"], metadata = params['metadata'])

            for param_idx, processed_param in enumerate(processed_params):
                solution = solution.replace(f"${param_idx}", str(processed_param))
                tests = tests.replace(f"${param_idx}", str(processed_param))

            tests = helper.replace_func_name(tests_template = tests, func_name = func_name)

            try:
                helper.run_test_suite(tests = tests, solution = solution)
            except Exception as e:
                print(type(e), e)
                print(f'{task_id} failed the tests when retrieved')
                # baseline_db.find_one_and_delete({"_id": task_id})
                break
        

  0%|          | 0/52 [00:00<?, ?it/s]

<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ1 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ2 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ3 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ4 failed the tests when retrieved


 13%|█▎        | 7/52 [00:00<00:00, 63.16it/s]

<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ5 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ6 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ7 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ8 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ9 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ10 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ11 failed the tests when retrieved
<class 'AttributeError'> 

 29%|██▉       | 15/52 [00:00<00:00, 69.48it/s]

...

<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ30 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ31 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ32 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ33 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ34 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ35 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ36 failed the tests when retrieved
<class 'AttributeErr

 79%|███████▉  | 41/52 [00:00<00:00, 77.08it/s]

<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ40 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ41 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ42 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ43 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ44 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ45 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ46 failed the tests when retrieved
<class 'AttributeErr

100%|██████████| 52/52 [00:00<00:00, 75.75it/s]

<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ48 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ49 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ50 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ51 failed the tests when retrieved
<class 'AttributeError'> 'TurbulenceBenchmarkHelper' object has no attribute 'run_test_suite'
TurbulenceQ52 failed the tests when retrieved
